In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("/content/DATA_Low_Rain.CSV")

In [ ]:
print(df.shape)
print(df.info())

(1357, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1357 entries, 0 to 1356
Data columns (total 20 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Time(ms)     1357 non-null   int64  
 1   Temp(C)      148 non-null    float64
 2   Humidity(%)  148 non-null    float64
 3   RainAnalog   1357 non-null   int64  
 4   RainDigital  1357 non-null   int64  
 5   SNR1         1357 non-null   int64  
 6   SNR2         1357 non-null   int64  
 7   SNR3         1357 non-null   int64  
 8   SNR4         1357 non-null   int64  
 9   SNR5         1357 non-null   int64  
 10  SNR6         1357 non-null   int64  
 11  SNR7         1357 non-null   int64  
 12  SNR8         1357 non-null   int64  
 13  SNR9         1357 non-null   int64  
 14  SNR10        1357 non-null   int64  
 15  SNR11        1357 non-null   int64  
 16  SNR12        1357 non-null   int64  
 17  SNR13        1357 non-null   int64  
 18  SNR14        1357 non-null   int64  


In [ ]:
(df == 0).sum()

,0
Time(ms),0
Temp(C),0
Humidity(%),0
RainAnalog,0
RainDigital,0
SNR1,101
SNR2,132
SNR3,186
SNR4,207
SNR5,271


In [ ]:
df['RainDigital'] = 1

In [ ]:
df['RainDigital'] = np.where(df['RainAnalog'] > 20, 1, 0)

In [ ]:
snr_cols = [col for col in df.columns if "SNR" in col]

for col in snr_cols:
    df.loc[df[col] > 100, col] = np.nan
    df.loc[df[col] < 10, col] = np.nan

In [ ]:
for col in snr_cols:
    df.loc[df[col] == 0, col] = np.nan

In [ ]:
df.loc[df['Temp(C)'] == 0, 'Temp(C)'] = np.nan
df.loc[df['Humidity(%)'] == 0, 'Humidity(%)'] = np.nan

In [ ]:
df = df.sort_values("Time(ms)")

df[snr_cols] = df[snr_cols].interpolate(method='linear')
df[['Temp(C)', 'Humidity(%)']] = df[['Temp(C)', 'Humidity(%)']].interpolate()

In [ ]:
df.fillna(method='bfill', inplace=True)
df.fillna(method='ffill', inplace=True)

/tmp/ipython-input-2053843447.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='bfill', inplace=True)
/tmp/ipython-input-2053843447.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)


In [ ]:
for col in snr_cols:
    df[col] = df[col].rolling(window=3, min_periods=1).mean()

In [ ]:
for col in snr_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

In [ ]:
df[snr_cols] = df[snr_cols].round().astype("Int64")

In [ ]:
df['Mean_SNR'] = df[snr_cols].mean(axis=1)

In [ ]:
df['SNR_Var'] = df[snr_cols].var(axis=1)

In [ ]:
df['SNR_Diff'] = df['Mean_SNR'].diff().fillna(0)

In [ ]:
df['Sat_Count'] = df[snr_cols].notna().sum(axis=1)

In [ ]:
df.to_csv("Low_Rain_Clean.csv", index=False)